# Hybrid validation, model selection and final holdout evaluation

This notebook tests whether plot similarity improves the collaborative model,
then freezes one model using validation mean Recall@10 across all model
families. Only after that decision does it read test labels and refit the
selected method on training plus validation interactions. Popularity is the
predeclared final-test benchmark.

The experiment uses the same per-user chronological 60/20/20 split, positive
threshold, candidate policy and macro averaging as the preceding notebooks.
All positive held-out movies remain in recall denominators. No test score is
used to choose a model or hybrid weight.


## Load canonical training and validation interactions

The shared loader checks source hashes and the split manifest. It reads no
test labels. Run notebooks 01–06 first so their split-specific artifacts and
per-user validation results are available.


In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10

# Resolve the project from either its root or any nested notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the project containing data/raw/ml-latest-small.")

raw_dir = data_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to write the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} does not match the split manifest.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]

train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]

def save_validation_results(results, filename):
    """Attach the split and evaluation contract to each per-user result row."""
    saved = results.copy()
    saved["split_id"] = SPLIT_ID
    saved["split_fingerprint"] = split_manifest["split_fingerprint"]
    saved["evaluation_split"] = "validation"
    saved["positive_threshold"] = POSITIVE_THRESHOLD
    saved["k"] = TOP_K
    saved.to_csv(processed_dir / filename, index=False)

def validate_result_provenance(results, label="results"):
    expected = {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }
    for column, value in expected.items():
        assert column in results and results[column].eq(value).all(), (
            f"{label}: missing or incompatible {column}; rerun its source notebook."
        )

print("Split:", SPLIT_ID)
print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))


## Rebuild the training-only CF and original TF-IDF representations

Binary likes define item similarity. The TF-IDF vocabulary and document
frequencies are learned only from plots belonging to training candidates.
Movies without plots stay in the candidate catalog with zero text evidence;
they are not removed from evaluation. Popularity counts provide the same
fallback and deterministic tie breaker used by the individual models.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

training_feedback = train_ratings[["userId", "movieId", "rating"]].copy()
training_feedback["liked"] = (training_feedback["rating"] >= POSITIVE_THRESHOLD).astype(int)
user_movie_matrix = (
    training_feedback.pivot(index="userId", columns="movieId", values="liked")
    .fillna(0).astype(int)
)
movie_user_matrix = user_movie_matrix.T
movie_popularity = (
    train_ratings.groupby("movieId").size().reset_index(name="training_rating_count")
    .merge(movies[["movieId", "title"]], on="movieId", validate="one_to_one")
    .sort_values(["training_rating_count", "movieId"], ascending=[False, True])
    .reset_index(drop=True)
)
assert set(movie_user_matrix.index) == set(movie_popularity["movieId"])

plot_catalog_manifest = json.loads(
    (processed_dir / "plot_catalog_manifest.json").read_text(encoding="utf-8")
)
assert plot_catalog_manifest["split_id"] == SPLIT_ID
assert plot_catalog_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
for filename in ("movie_plot_catalog.csv", "training_plot_catalog.csv"):
    assert file_sha256(processed_dir / filename) == plot_catalog_manifest["files"][filename]["sha256"]
training_plot_catalog = pd.read_csv(processed_dir / "training_plot_catalog.csv")
assert training_plot_catalog["movieId"].is_unique
assert set(training_plot_catalog["movieId"]) == set(train_ratings["movieId"])
training_plot_catalog["has_plot"] = training_plot_catalog["plot"].fillna("").str.strip().ne("")
text_movies = (
    training_plot_catalog.loc[training_plot_catalog["has_plot"]]
    .sort_values("movieId").reset_index(drop=True).copy()
)
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True, stop_words="english", min_df=2, norm="l2"
)
plot_tfidf = tfidf_vectorizer.fit_transform(text_movies["plot"])
print("CF movie/user matrix:", movie_user_matrix.shape)
print("Training movies with plots:", len(text_movies))
print("Plot TF-IDF matrix:", plot_tfidf.shape)


## Keep model scores separate before blending

For each user, CF and plot scores average similarity to that user's liked
training movies. Every rated training movie is excluded, regardless of its
rating. Hybrid weights combine scores after scaling each component by its
maximum across all eligible candidates for that user. This normalization is
a ranking heuristic, not a calibrated probability. When plots provide no
evidence, the ranking preserves CF; zero plot weight must reproduce CF
exactly, including ties. Returned component columns support the inspection
table without recalculating a different normalization.


In [ ]:
def prepare_hybrid_scores(
    user_id, training_data, cf_matrix, popularity, text_movies, text_matrix
):
    """Score the same unseen candidates using CF and plot evidence separately."""
    history = training_data.loc[training_data["userId"] == user_id]
    liked_ids = history.loc[history["rating"] >= POSITIVE_THRESHOLD, "movieId"]
    candidates = popularity.loc[
        ~popularity["movieId"].isin(history["movieId"])
    ].copy()
    candidates["cf_score"] = 0.0
    candidates["text_score"] = 0.0
    if liked_ids.empty:
        return candidates

    cf_values = cosine_similarity(cf_matrix.loc[liked_ids], cf_matrix).mean(axis=0)
    candidates["cf_score"] = candidates["movieId"].map(
        pd.Series(cf_values, index=cf_matrix.index)
    ).fillna(0.0)
    text_rows = np.flatnonzero(text_movies["movieId"].isin(liked_ids).to_numpy())
    if len(text_rows):
        text_values = cosine_similarity(text_matrix[text_rows], text_matrix).mean(axis=0)
        candidates["text_score"] = candidates["movieId"].map(
            pd.Series(text_values, index=text_movies["movieId"])
        ).fillna(0.0)
    return candidates

def rank_hybrid(candidate_scores, plot_weight=0.10, n=TOP_K):
    """Blend per-user max-scaled scores, preserving CF when plots add no evidence."""
    if not 0 <= plot_weight <= 1:
        raise ValueError("plot_weight must be between 0 and 1.")
    ranked = candidate_scores.copy()
    text_max = ranked["text_score"].max()
    if ranked.empty or plot_weight == 0 or text_max <= 0:
        ranked["cf_part"] = ranked["cf_score"]
        ranked["plot_part"] = 0.0
    else:
        cf_max = ranked["cf_score"].max()
        ranked["cf_part"] = (
            (1 - plot_weight) * ranked["cf_score"] / cf_max if cf_max > 0 else 0.0
        )
        ranked["plot_part"] = plot_weight * ranked["text_score"] / text_max
    ranked["score"] = ranked["cf_part"] + ranked["plot_part"]
    return (
        ranked.sort_values(
            ["score", "training_rating_count", "movieId"],
            ascending=[False, False, True],
        ).head(n).reset_index(drop=True)
    )


## Evaluate the declared weight grid on validation only

The tested weights are fixed here before final testing. Compute component
scores once per user, then evaluate each blend on the same user and positive
movies. Users without a usable liked history remain eligible and receive
the documented fallback. Assertions check unique recommendations, catalog
membership, list length and exclusion of seen movies.


In [ ]:
plot_weights = [0.0, 0.10, 0.25, 0.50]
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
]
if validation_positives.empty:
    raise ValueError("There are no positive validation ratings to evaluate.")
training_users = set(train_ratings["userId"])
training_candidates = set(train_ratings["movieId"])
evaluation_rows = []
for user_id, positives in validation_positives.groupby("userId"):
    scores = prepare_hybrid_scores(
        user_id, train_ratings, movie_user_matrix, movie_popularity, text_movies, plot_tfidf
    )
    positive_ids = set(positives["movieId"])
    seen_ids = set(train_ratings.loc[train_ratings["userId"].eq(user_id), "movieId"])
    for weight in plot_weights:
        recommendations = rank_hybrid(scores, plot_weight=weight, n=TOP_K)
        recommended_ids = set(recommendations["movieId"])
        assert recommendations["movieId"].is_unique
        assert len(recommendations) <= TOP_K
        assert recommended_ids.isdisjoint(seen_ids)
        assert recommended_ids.issubset(training_candidates)
        hits = len(positive_ids & recommended_ids)
        evaluation_rows.append({
            "plot_weight": weight, "userId": user_id,
            "positive_movie_count": len(positive_ids), "hits_at_10": hits,
            "recall_at_10": hits / len(positive_ids),
            "has_training_history": user_id in training_users,
        })
hybrid_results = pd.DataFrame(evaluation_rows)
assert not hybrid_results.duplicated(["plot_weight", "userId"]).any()
hybrid_results["plot_catalog_sha256"] = file_sha256(processed_dir / "training_plot_catalog.csv")
save_validation_results(hybrid_results, "hybrid_validation_results.csv")


## Verify the CF control and summarize the grid

The zero-weight control must match notebook 02's per-user hits and recall.
Its saved results must also match the current split fingerprint. This check
detects silent changes to scoring, tie rules or the validation population.
The best tested blend is selected by mean recall, with lower plot weight
winning an exact tie.


In [ ]:
saved_cf = pd.read_csv(processed_dir / "item_based_validation_results.csv")
validate_result_provenance(saved_cf, "Item-based CF")
baseline_check = hybrid_results.loc[hybrid_results["plot_weight"].eq(0)].merge(
    saved_cf, on="userId", how="outer", suffixes=("_new", "_saved"),
    indicator=True, validate="one_to_one",
)
assert baseline_check["_merge"].eq("both").all()
for column in ("positive_movie_count", "hits_at_10", "has_training_history"):
    assert baseline_check[f"{column}_new"].eq(baseline_check[f"{column}_saved"]).all()
assert np.allclose(baseline_check["recall_at_10_new"], baseline_check["recall_at_10_saved"])
print("Zero plot weight reproduces the saved CF baseline.")

hybrid_comparison = (
    hybrid_results.groupby("plot_weight")
    .agg(users=("userId", "size"), mean_recall_at_10=("recall_at_10", "mean"),
         total_hits_at_10=("hits_at_10", "sum"))
    .reset_index()
    .sort_values(["mean_recall_at_10", "plot_weight"], ascending=[False, True])
    .reset_index(drop=True)
)
best_hybrid_weight = float(hybrid_comparison.iloc[0]["plot_weight"])
display(hybrid_comparison.assign(
    mean_recall_at_10=hybrid_comparison["mean_recall_at_10"].map("{:.4%}".format)
))
save_validation_results(hybrid_comparison, "hybrid_validation_comparison.csv")


## Inspect which users and recommendations changed

Pair each user's selected-blend score with the zero-weight control. Inspect
the two users with the largest absolute recall change, breaking ties by user
ID, rather than retaining examples chosen under the previous split. An
empty change table is valid when the selected weight is zero or both
rankings have the same top ten. This inspection is descriptive and does
not introduce additional candidate weights.


In [ ]:
columns = ["userId", "positive_movie_count", "hits_at_10", "recall_at_10"]
cf_results = hybrid_results.loc[hybrid_results["plot_weight"].eq(0), columns]
blend_results = hybrid_results.loc[hybrid_results["plot_weight"].eq(best_hybrid_weight), columns]
hybrid_user_comparison = cf_results.merge(
    blend_results, on="userId", suffixes=("_cf", "_hybrid"), validate="one_to_one"
)
assert hybrid_user_comparison["positive_movie_count_cf"].eq(
    hybrid_user_comparison["positive_movie_count_hybrid"]
).all()
hybrid_user_comparison["recall_change"] = (
    hybrid_user_comparison["recall_at_10_hybrid"] - hybrid_user_comparison["recall_at_10_cf"]
)
hybrid_user_comparison["plot_weight"] = best_hybrid_weight
print("Best tested weight:", best_hybrid_weight)
print("Users improved:", hybrid_user_comparison["recall_change"].gt(0).sum())
print("Users worsened:", hybrid_user_comparison["recall_change"].lt(0).sum())
print("Users unchanged:", hybrid_user_comparison["recall_change"].eq(0).sum())
save_validation_results(hybrid_user_comparison, "hybrid_selected_weight_vs_cf_users.csv")

def inspect_hybrid_changes(user_id, plot_weight):
    candidates = prepare_hybrid_scores(
        user_id, train_ratings, movie_user_matrix, movie_popularity, text_movies, plot_tfidf
    )
    cf_ranked = rank_hybrid(candidates, plot_weight=0, n=len(candidates))
    blend_ranked = rank_hybrid(candidates, plot_weight=plot_weight, n=len(candidates))
    cf_top = set(cf_ranked.head(TOP_K)["movieId"])
    blend_top = set(blend_ranked.head(TOP_K)["movieId"])
    changes = blend_ranked.loc[blend_ranked["movieId"].isin(cf_top ^ blend_top)].copy()
    cf_ranks = pd.Series(np.arange(1, len(cf_ranked)+1), index=cf_ranked["movieId"])
    blend_ranks = pd.Series(np.arange(1, len(blend_ranked)+1), index=blend_ranked["movieId"])
    changes["cf_rank"] = changes["movieId"].map(cf_ranks)
    changes["hybrid_rank"] = changes["movieId"].map(blend_ranks)
    changes["change"] = np.where(changes["movieId"].isin(blend_top), "Entered", "Left")
    changes["has_plot"] = changes["movieId"].isin(text_movies["movieId"])
    positive_ids = set(validation_positives.loc[validation_positives["userId"].eq(user_id), "movieId"])
    changes["positive_in_validation"] = changes["movieId"].isin(positive_ids)
    return changes[["change", "movieId", "title", "has_plot", "cf_rank", "hybrid_rank",
                    "cf_part", "plot_part", "positive_in_validation"]].sort_values(
                        ["change", "hybrid_rank"]
                    )

inspection_ids = (
    hybrid_user_comparison.assign(magnitude=lambda frame: frame["recall_change"].abs())
    .sort_values(["magnitude", "userId"], ascending=[False, True])["userId"].head(2)
)
for user_id in inspection_ids:
    changes = inspect_hybrid_changes(user_id, best_hybrid_weight)
    print("User:", user_id)
    display(changes)
    changes.to_csv(processed_dir / f"hybrid_changes_user_{user_id}.csv", index=False)


## Freeze the model before accessing test labels

Compare the seven individual model variants with every nonzero hybrid weight.
The zero-weight hybrid duplicates CF and is not counted twice. Exact mean
recall ties use the declared family order below, preferring lightweight
baselines before more involved feature pipelines; exact blend ties prefer
less plot contribution. This deterministic rule is part of the experiment,
not a decision based on test performance.

The selection record stores the winning family, weight, evaluation contract
and split fingerprint before the test-reading cell. Once test results exist,
the notebook rejects changing that frozen choice in the same experiment.
Rerunning an unchanged choice is a reproducibility check, not a fresh
independent evaluation.


In [ ]:
result_files = {
    "Popularity": "popularity_validation_results.csv",
    "Item-based CF": "item_based_validation_results.csv",
    "Genres": "genre_validation_results.csv",
    "Original TF-IDF": "tfidf_validation_results.csv",
    "Log-TF TF-IDF": "log_tfidf_validation_results.csv",
    "Embeddings": "embedding_validation_results.csv",
    "Adjusted embeddings": "adjusted_embedding_validation_results.csv",
}
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
]
expected_counts = (
    validation_positives.groupby("userId")["movieId"].nunique().sort_index()
)
if expected_counts.empty:
    raise ValueError("There are no positive validation ratings to evaluate.")

def checked_validation_results(filename, label):
    results = pd.read_csv(processed_dir / filename)
    validate_result_provenance(results, label)
    if label in {"Original TF-IDF", "Log-TF TF-IDF", "Embeddings", "Adjusted embeddings"}:
        catalog_path = processed_dir / "training_plot_catalog.csv"
        catalog_manifest = json.loads(
            (processed_dir / "plot_catalog_manifest.json").read_text(encoding="utf-8")
        )
        assert catalog_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
        expected_catalog_hash = catalog_manifest["files"][catalog_path.name]["sha256"]
        assert file_sha256(catalog_path) == expected_catalog_hash
        assert "plot_catalog_sha256" in results and results["plot_catalog_sha256"].eq(
            expected_catalog_hash
        ).all(), f"{label}: plot catalog changed; rerun its source notebook"
    assert results["userId"].is_unique, f"{label}: repeated evaluated users"
    results = results.set_index("userId").sort_index()
    assert results.index.equals(expected_counts.index), f"{label}: evaluated users differ"
    assert results["positive_movie_count"].eq(expected_counts).all(), (
        f"{label}: recall denominators differ from canonical validation positives"
    )
    assert results["hits_at_10"].between(0, TOP_K).all(), label
    assert results["hits_at_10"].le(results["positive_movie_count"]).all(), label
    assert results["hits_at_10"].mod(1).eq(0).all(), label
    assert np.allclose(
        results["recall_at_10"],
        results["hits_at_10"] / results["positive_movie_count"],
    ), f"{label}: saved recall is inconsistent with hits and positives"
    expected_history = results.index.isin(train_ratings["userId"])
    assert "has_training_history" in results, label
    assert results["has_training_history"].eq(expected_history).all(), label
    return results

evaluations = {
    label: checked_validation_results(filename, label)
    for label, filename in result_files.items()
}

# Exact score ties use this declared preference order; blend ties use lower plot weight.
model_preference_order = [
    "Popularity", "Genres", "Item-based CF", "Original TF-IDF", "Log-TF TF-IDF",
    "Embeddings", "Adjusted embeddings", "Hybrid",
]
selection_rows = [
    {"model": label, "plot_weight": 0.0,
     "mean_recall_at_10": float(results["recall_at_10"].mean()), "users": len(results)}
    for label, results in evaluations.items()
]
for weight, results in hybrid_results.groupby("plot_weight"):
    if weight == 0:
        continue  # Identical to the already included Item-based CF model.
    current = results.set_index("userId").sort_index()
    assert current.index.equals(expected_counts.index)
    assert current["positive_movie_count"].eq(expected_counts).all()
    selection_rows.append({
        "model": "Hybrid", "plot_weight": float(weight),
        "mean_recall_at_10": float(current["recall_at_10"].mean()), "users": len(current),
    })
selection_table = pd.DataFrame(selection_rows)
selection_table["preference"] = selection_table["model"].map(
    {model: rank for rank, model in enumerate(model_preference_order)}
)
selection_table = selection_table.sort_values(
    ["mean_recall_at_10", "preference", "plot_weight"], ascending=[False, True, True]
).reset_index(drop=True)
winner = selection_table.iloc[0]
selected_config = {
    "model": str(winner["model"]),
    "plot_weight": float(winner["plot_weight"]),
    "validation_mean_recall_at_10": float(winner["mean_recall_at_10"]),
    "selection_metric": "macro mean Recall@10",
    "split_id": SPLIT_ID,
    "split_fingerprint": split_manifest["split_fingerprint"],
    "positive_threshold": POSITIVE_THRESHOLD,
    "k": TOP_K,
    "model_preference_order": model_preference_order,
    "hybrid_weights_tested": plot_weights,
    "refit_policy": "training plus validation interactions",
    "candidate_policy": split_manifest["candidate_policy"],
    "recall_denominator": split_manifest["recall_denominator"],
    "movie_plot_catalog_sha256": file_sha256(processed_dir / "movie_plot_catalog.csv"),
}
if selected_config["model"] in {"Embeddings", "Adjusted embeddings"}:
    embedding_manifest = json.loads(
        (processed_dir / "embedding_manifest.json").read_text(encoding="utf-8")
    )
    assert embedding_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
    selected_config["embedding_model_name"] = embedding_manifest["model_name"]
    selected_config["embedding_model_revision"] = embedding_manifest.get("model_revision")
selection_path = processed_dir / "frozen_selection.json"
if selection_path.is_file() and (processed_dir / "final_test_results.csv").is_file():
    previous = json.loads(selection_path.read_text(encoding="utf-8"))
    for key in ("model", "plot_weight", "split_fingerprint", "positive_threshold", "k",
                "movie_plot_catalog_sha256", "embedding_model_name", "embedding_model_revision"):
        if previous.get(key) != selected_config.get(key):
            raise ValueError("The test partition was already evaluated for a different selection. "
                             "Do not use its outcomes to choose a replacement model.")
selection_path.write_text(json.dumps(selected_config, indent=2) + "\n", encoding="utf-8")
save_validation_results(selection_table, "final_validation_selection.csv")
display(selection_table.drop(columns="preference").assign(
    mean_recall_at_10=selection_table["mean_recall_at_10"].map("{:.4%}".format)
))
print("Frozen model:", selected_config["model"])
print("Frozen plot weight:", selected_config["plot_weight"])


## Reuse the same plot embedding implementation

The helper below is copied from notebook 05 rather than imported from a
separately maintained project file. It tokenizes complete plots into
nonoverlapping model-length chunks, embeds each chunk using the pretrained
model's normal pooling, normalizes chunk vectors, and combines them with
content-token-count weights before normalizing each movie vector. Defining
the helper does not load a language model or encode plots. Those operations
run only if an embedding family wins validation.


In [ ]:
"""Reusable version of notebook05's fixed plot embedding recipe."""
import numpy as np
import pandas as pd


def embed_chunk_batch(token_id_lists, model):
    """Embed BERT-style chunks with model pooling and unit-length outputs."""
    import torch
    import torch.nn.functional as F

    tokenizer = model.tokenizer
    if not token_id_lists:
        return np.empty((0, model.get_sentence_embedding_dimension()), dtype=np.float32)
    sequences = [
        [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]
        for ids in token_id_lists
    ]
    batch_length = max(map(len, sequences))
    if batch_length > model.max_seq_length:
        raise ValueError("A chunk exceeds the model's input limit.")
    input_ids = torch.full(
        (len(sequences), batch_length), tokenizer.pad_token_id,
        dtype=torch.long, device=model.device,
    )
    attention_mask = torch.zeros_like(input_ids)
    for row, ids in enumerate(sequences):
        input_ids[row, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=model.device)
        attention_mask[row, :len(ids)] = 1
    features = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": torch.zeros_like(input_ids),
    }
    model.eval()
    with torch.inference_mode():
        output = model(features)
        return F.normalize(output["sentence_embedding"], p=2, dim=1).cpu().numpy()


def embed_movie_plots(catalog, model, batch_size=16):
    """Return unit vectors in catalog row order, preserving all plot tokens.

    The caller supplies unique movieId rows with nonblank plot strings.
    Chunk vectors use the pretrained model's normal pooling; aggregation
    weights are content-token counts. No interaction labels are consumed.
    """
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    if not catalog["movieId"].is_unique:
        raise ValueError("catalog movieId values must be unique.")
    if catalog["plot"].fillna("").str.strip().eq("").any():
        raise ValueError("Only movies with nonblank plots can be embedded.")
    dimensions = model.get_sentence_embedding_dimension()
    if catalog.empty:
        return np.empty((0, dimensions), dtype=np.float32)
    tokenizer = model.tokenizer
    content_limit = model.max_seq_length - tokenizer.num_special_tokens_to_add(pair=False)
    if content_limit < 1:
        raise ValueError("The input limit leaves no room for content tokens.")
    chunks = []
    owners = []
    lengths = []
    for movie_row, plot in enumerate(catalog["plot"]):
        ids = tokenizer(
            plot, add_special_tokens=False, truncation=False, verbose=False
        )["input_ids"]
        if not ids:
            raise ValueError(f"Plot at catalog row {movie_row} has no content tokens.")
        for start in range(0, len(ids), content_limit):
            chunk = ids[start:start + content_limit]
            chunks.append(chunk)
            owners.append(movie_row)
            lengths.append(len(chunk))
    chunk_vectors = np.empty((len(chunks), dimensions), dtype=np.float32)
    for start in range(0, len(chunks), batch_size):
        stop = min(start + batch_size, len(chunks))
        chunk_vectors[start:stop] = embed_chunk_batch(chunks[start:stop], model)
    owners = np.asarray(owners)
    lengths = np.asarray(lengths)
    movie_vectors = np.empty((len(catalog), dimensions), dtype=np.float32)
    for movie_row in range(len(catalog)):
        positions = np.flatnonzero(owners == movie_row)
        movie_vectors[movie_row] = np.average(
            chunk_vectors[positions], axis=0, weights=lengths[positions]
        )
    norms = np.linalg.norm(movie_vectors, axis=1)
    if not np.isfinite(movie_vectors).all() or not (norms > 0).all():
        raise ValueError("Movie vector aggregation produced nonfinite or zero vectors.")
    movie_vectors /= norms[:, None]
    return movie_vectors


## Refit the frozen family with training and validation interactions

The candidate catalog now contains movies observed in either fitting
partition. Rebuild fitted counts and collaborative, genre or TF-IDF features
from those interactions and candidate plots as required by the selected
family. A selected embedding model uses the same pretrained encoder and
chunk aggregation as notebook 05; centering uses the refitted catalog mean.
All of a user's training and validation ratings become seen-item exclusions.
No test interaction contributes to representations, profiles or popularity.

The final engine supports every family considered during validation, so the
test does not silently retain the winner from the earlier global-date split.


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix


def fit_final_model(configuration, fitting_data, movie_catalog):
    """Fit a frozen model family using only the supplied interactions."""
    family = configuration["model"]
    allowed = {
        "Popularity", "Item-based CF", "Genres", "Original TF-IDF",
        "Log-TF TF-IDF", "Embeddings", "Adjusted embeddings", "Hybrid",
    }
    if family not in allowed:
        raise ValueError(f"Unsupported selected model: {family}")
    assert not fitting_data.duplicated(["userId", "movieId"]).any()
    popularity = (
        fitting_data.groupby("movieId").size()
        .reset_index(name="training_rating_count")
        .merge(movie_catalog[["movieId", "title"]], on="movieId", validate="one_to_one")
        .sort_values(["training_rating_count", "movieId"], ascending=[False, True])
        .reset_index(drop=True)
    )
    state = {
        "family": family,
        "configuration": configuration.copy(),
        "fitting_data": fitting_data.copy(),
        "popularity": popularity,
    }
    candidate_ids = set(popularity["movieId"])
    if family in {"Item-based CF", "Hybrid"}:
        feedback = fitting_data[["userId", "movieId", "rating"]].copy()
        feedback["liked"] = feedback["rating"].ge(POSITIVE_THRESHOLD).astype(int)
        state["cf_matrix"] = (
            feedback.pivot(index="userId", columns="movieId", values="liked")
            .fillna(0).astype(int).T
        )
        assert set(state["cf_matrix"].index) == candidate_ids
    if family == "Genres":
        candidates = movie_catalog.loc[movie_catalog["movieId"].isin(candidate_ids)].copy()
        candidates = candidates.sort_values("movieId").reset_index(drop=True)
        labels = candidates["genres"].fillna("(no genres listed)").map(
            lambda value: [] if value == "(no genres listed)" else value.split("|")
        )
        encoder = MultiLabelBinarizer()
        state["genre_matrix"] = pd.DataFrame(
            encoder.fit_transform(labels), index=candidates["movieId"], columns=encoder.classes_
        )
    if family in {"Original TF-IDF", "Log-TF TF-IDF", "Embeddings", "Adjusted embeddings", "Hybrid"}:
        # Rebuild the catalog membership after adding validation interactions.
        # Text itself is external metadata, never a held-out rating feature.
        full_catalog_path = processed_dir / "movie_plot_catalog.csv"
        catalog_manifest = json.loads(
            (processed_dir / "plot_catalog_manifest.json").read_text(encoding="utf-8")
        )
        assert catalog_manifest["split_id"] == SPLIT_ID
        assert catalog_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
        assert file_sha256(full_catalog_path) == catalog_manifest["files"][full_catalog_path.name]["sha256"]
        if "movie_plot_catalog_sha256" in configuration:
            assert file_sha256(full_catalog_path) == configuration["movie_plot_catalog_sha256"]
        full_catalog = pd.read_csv(full_catalog_path)
        assert full_catalog["movieId"].is_unique
        assert set(full_catalog["movieId"]) == set(movie_catalog["movieId"])
        plot_catalog = full_catalog.loc[full_catalog["movieId"].isin(candidate_ids)].copy()
        plot_catalog["has_plot"] = plot_catalog["plot"].fillna("").str.strip().ne("")
        state["plot_catalog"] = plot_catalog
        text = (
            plot_catalog.loc[plot_catalog["has_plot"]]
            .sort_values("movieId").reset_index(drop=True).copy()
        )
        state["text_movies"] = text
        if family in {"Original TF-IDF", "Log-TF TF-IDF", "Hybrid"}:
            vectorizer = TfidfVectorizer(
                lowercase=True, stop_words="english", min_df=2, norm="l2",
                sublinear_tf=(family == "Log-TF TF-IDF"),
            )
            if text.empty:
                matrix = csr_matrix((0, 0), dtype=float)
            else:
                try:
                    matrix = vectorizer.fit_transform(text["plot"])
                except ValueError as error:
                    if not any(message in str(error) for message in (
                        "empty vocabulary", "After pruning", "max_df corresponds"
                    )):
                        raise
                    # An empty usable vocabulary contains no text evidence.
                    matrix = csr_matrix((len(text), 0), dtype=float)
            state["text_matrix"] = matrix
            state["vectorizer"] = vectorizer
        else:
            from sentence_transformers import SentenceTransformer
            embedding_manifest = json.loads(
                (processed_dir / "embedding_manifest.json").read_text(encoding="utf-8")
            )
            assert embedding_manifest["split_fingerprint"] == split_manifest["split_fingerprint"]
            assert embedding_manifest["model_name"] == "sentence-transformers/all-MiniLM-L6-v2"
            for key, manifest_key in (
                ("embedding_model_name", "model_name"),
                ("embedding_model_revision", "model_revision"),
            ):
                if key in configuration:
                    assert configuration[key] == embedding_manifest.get(manifest_key), (
                        "Embedding encoder metadata changed after the selection was frozen."
                    )
            # Reuse the exact pretrained snapshot recorded during validation.
            model_options = {"device": "cpu"}
            if embedding_manifest.get("model_revision") is not None:
                model_options["revision"] = embedding_manifest["model_revision"]
            encoder = SentenceTransformer(embedding_manifest["model_name"], **model_options)
            vectors = embed_movie_plots(text, encoder, batch_size=16)
            assert len(vectors) == len(text)
            assert np.isfinite(vectors).all()
            state["movie_embeddings"] = vectors
            state["catalog_mean_vector"] = (
                vectors.mean(axis=0) if len(vectors) else np.zeros(vectors.shape[1])
            )
    return state


def recommend_final_model(user_id, state, n=TOP_K):
    """Use the selected validation algorithm with refitted features and history."""
    history = state["fitting_data"].loc[state["fitting_data"]["userId"].eq(user_id)]
    liked_ids = history.loc[history["rating"].ge(POSITIVE_THRESHOLD), "movieId"]
    candidates = state["popularity"].loc[
        ~state["popularity"]["movieId"].isin(history["movieId"])
    ].copy()
    candidates["score"] = 0.0
    family = state["family"]
    if family == "Hybrid":
        text_matrix = state["text_matrix"]
        if text_matrix.shape[1] > 0:
            scores = prepare_hybrid_scores(
                user_id, state["fitting_data"], state["cf_matrix"],
                state["popularity"], state["text_movies"], text_matrix,
            )
        else:
            # Preserve the same CF fallback when no text evidence is available.
            cf_state = dict(state, family="Item-based CF")
            return recommend_final_model(user_id, cf_state, n=n)
        return rank_hybrid(scores, plot_weight=float(state["configuration"]["plot_weight"]), n=n)
    if not liked_ids.empty:
        if family in {"Item-based CF", "Genres"}:
            matrix = state["cf_matrix" if family == "Item-based CF" else "genre_matrix"]
            usable = liked_ids[liked_ids.isin(matrix.index)]
            if len(usable) and matrix.shape[1] > 0:
                values = cosine_similarity(matrix.loc[usable], matrix).mean(axis=0)
                scores = pd.Series(values, index=matrix.index)
                candidates["score"] = candidates["movieId"].map(scores).fillna(0.0)
        elif family in {"Original TF-IDF", "Log-TF TF-IDF"}:
            text = state["text_movies"]
            rows = np.flatnonzero(text["movieId"].isin(liked_ids).to_numpy())
            if len(rows) and state["text_matrix"].shape[1] > 0:
                values = cosine_similarity(
                    state["text_matrix"][rows], state["text_matrix"]
                ).mean(axis=0)
                scores = pd.Series(values, index=text["movieId"])
                candidates["score"] = candidates["movieId"].map(scores).fillna(0.0)
        elif family in {"Embeddings", "Adjusted embeddings"}:
            text = state["text_movies"]
            mask = text["movieId"].isin(liked_ids).to_numpy()
            if mask.any():
                vectors = state["movie_embeddings"]
                profile = vectors[mask].mean(axis=0)
                if family == "Adjusted embeddings":
                    profile = profile - state["catalog_mean_vector"]
                # Preserve signed scores and the unnormalized mean profile.
                scores = pd.Series(vectors @ profile, index=text["movieId"])
                candidates["score"] = candidates["movieId"].map(scores).fillna(0.0)
    return (
        candidates.sort_values(
            ["score", "training_rating_count", "movieId"], ascending=[False, False, True]
        ).head(n).reset_index(drop=True)
    )


## Evaluate the frozen winner and popularity on the held-out test partition

Read and hash-check the canonical test file only after selection is frozen.
Verify rating-pair separation and chronology within each user's history;
there is no single global date boundary in this split. Each model uses the
same positive test users and all their positive movies in the denominator.
The selected-model label remains separate from the popularity benchmark even
when popularity itself wins validation, making a zero paired difference
explicit. Test results are for assessment; they must not drive further
tuning on this same held-out partition.


In [ ]:
# Read back the persisted choice before opening any test labels.
selected_config = json.loads((processed_dir / "frozen_selection.json").read_text(encoding="utf-8"))
assert selected_config["split_fingerprint"] == split_manifest["split_fingerprint"]
assert selected_config["positive_threshold"] == POSITIVE_THRESHOLD
assert selected_config["k"] == TOP_K

final_fit_ratings = pd.concat([train_ratings, validation_ratings], ignore_index=True)
final_fit_ratings = final_fit_ratings.sort_values(["userId", "timestamp", "movieId"])
assert not final_fit_ratings.duplicated(["userId", "movieId"]).any()

# The hash binds the holdout to notebook01's exact partition.
test_path = processed_dir / "test_ratings.csv"
test_expected = split_manifest["files"][test_path.name]
assert file_sha256(test_path) == test_expected["sha256"]
test_ratings = pd.read_csv(test_path, parse_dates=["rated_at"])
assert len(test_ratings) == test_expected["rows"]
assert not test_ratings.duplicated(["userId", "movieId"]).any()
fit_pairs = pd.MultiIndex.from_frame(final_fit_ratings[["userId", "movieId"]])
test_pairs = pd.MultiIndex.from_frame(test_ratings[["userId", "movieId"]])
assert fit_pairs.intersection(test_pairs).empty

fit_latest = final_fit_ratings.groupby("userId")["rated_at"].max()
test_earliest = test_ratings.groupby("userId")["rated_at"].min()
assert set(test_earliest.index).issubset(fit_latest.index)
assert (fit_latest.reindex(test_earliest.index) <= test_earliest).all()

final_model = fit_final_model(selected_config, final_fit_ratings, movies)
# Popularity is the fixed benchmark, refitted on the same interactions.
final_popularity = final_model["popularity"]
final_candidate_ids = set(final_popularity["movieId"])
fitting_users = set(final_fit_ratings["userId"])
test_positives = test_ratings.loc[test_ratings["rating"].ge(POSITIVE_THRESHOLD)]
if test_positives.empty:
    raise ValueError("The test partition has no positive ratings; Recall@10 is undefined.")

test_rows = []
for user_id, positives in test_positives.groupby("userId", sort=True):
    positive_ids = set(positives["movieId"])
    seen_ids = set(final_fit_ratings.loc[final_fit_ratings["userId"].eq(user_id), "movieId"])
    recommendations_by_role = {
        "Selected model": recommend_final_model(user_id, final_model),
        "Popularity": final_popularity.loc[
            ~final_popularity["movieId"].isin(seen_ids)
        ].head(TOP_K).reset_index(drop=True),
    }
    for role, recommendations in recommendations_by_role.items():
        recommended_ids = set(recommendations["movieId"])
        assert recommendations["movieId"].is_unique
        assert len(recommendations) <= TOP_K
        assert recommended_ids.issubset(final_candidate_ids)
        assert recommended_ids.isdisjoint(seen_ids)
        hits = len(positive_ids.intersection(recommended_ids))
        test_rows.append({
            "model": role,
            "model_family": selected_config["model"] if role == "Selected model" else "Popularity",
            "userId": user_id,
            "positive_movie_count": len(positive_ids),
            "unavailable_positive_movie_count": len(positive_ids - final_candidate_ids),
            "hits_at_10": hits,
            "recall_at_10": hits / len(positive_ids),
            "has_training_history": user_id in fitting_users,
        })

test_results = pd.DataFrame(test_rows)
assert not test_results.duplicated(["model", "userId"]).any()
test_results["split_id"] = SPLIT_ID
test_results["split_fingerprint"] = split_manifest["split_fingerprint"]
test_results["evaluation_split"] = "test"
test_results["positive_threshold"] = POSITIVE_THRESHOLD
test_results["k"] = TOP_K
test_results["selected_plot_weight"] = selected_config.get("plot_weight")
test_results.to_csv(processed_dir / "final_test_results.csv", index=False)
print("Final model:", selected_config["model"])
print("Final fitting ratings:", len(final_fit_ratings))
print("Test ratings:", len(test_ratings))
print("Test users with positive ratings:", test_positives["userId"].nunique())
print("Test users without positive ratings:", test_ratings["userId"].nunique() - test_positives["userId"].nunique())


## Report final performance and uncertainty in the paired gain

Report macro mean Recall@10 for the winner and the predefined benchmark.
Bootstrap users' paired recall differences, preserving the pairing, with a
fixed random seed and a percentile 95% interval. Empty, one-user and constant
differences receive explicit handling. This interval describes variation
across users in this dataset; it does not account for item/user dependence,
all model-selection uncertainty, future drift or repeated experimentation.

The split preserves each user's chronology but can include another user's
later training interaction. Plot snapshots and pretrained embeddings also
need not have existed at historical rating times. Interpret results as an
offline existing-user retrieval benchmark, not a backtest at one deployment
date. No old-split score or winner is carried into the conclusion.


In [ ]:
test_summary = (
    test_results.groupby(["model", "model_family"], sort=False)
    .agg(users=("userId", "nunique"), mean_recall_at_10=("recall_at_10", "mean"),
         total_hits_at_10=("hits_at_10", "sum"),
         unavailable_positives=("unavailable_positive_movie_count", "sum"))
    .reset_index()
)
test_history_summary = (
    test_results.groupby(["model", "has_training_history"], sort=False)
    .agg(users=("userId", "nunique"), mean_recall_at_10=("recall_at_10", "mean"),
         total_hits_at_10=("hits_at_10", "sum"))
    .reset_index()
)
for summary in (test_summary, test_history_summary):
    summary["split_id"] = SPLIT_ID
    summary["split_fingerprint"] = split_manifest["split_fingerprint"]
    summary["evaluation_split"] = "test"
test_summary.to_csv(processed_dir / "test_model_comparison.csv", index=False)
test_history_summary.to_csv(processed_dir / "test_results_by_history.csv", index=False)
for summary in (test_summary, test_history_summary):
    shown = summary.copy()
    shown["mean_recall_at_10"] = shown["mean_recall_at_10"].map("{:.4%}".format)
    display(shown)

# Resample users, retaining their paired selected-model and baseline recalls.
paired = test_results.pivot(index="userId", columns="model", values="recall_at_10")
assert paired[["Selected model", "Popularity"]].notna().all().all()
paired["recall_gain"] = paired["Selected model"] - paired["Popularity"]
history_flags = (
    test_results.loc[test_results["model"].eq("Selected model")]
    .set_index("userId")["has_training_history"]
)
paired["has_training_history"] = history_flags


def paired_mean_interval(values, seed=42, resamples=10000):
    """Percentile bootstrap for a paired mean; guard degenerate cohorts."""
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Bootstrap inputs must be finite per-user differences.")
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    mean = float(values.mean())
    if len(values) == 1 or np.ptp(values) == 0:
        return mean, mean, mean
    rng = np.random.default_rng(seed)
    boot_means = np.empty(resamples)
    # Bounded batches avoid an unnecessary resamples-by-users allocation.
    for start in range(0, resamples, 256):
        stop = min(start + 256, resamples)
        positions = rng.integers(0, len(values), size=(stop - start, len(values)))
        boot_means[start:stop] = values[positions].mean(axis=1)
    low, high = np.quantile(boot_means, [0.025, 0.975])
    return mean, float(low), float(high)


bootstrap_rows = []
for label, differences in {
    "All test users": paired["recall_gain"],
    "With fitting history": paired.loc[paired["has_training_history"], "recall_gain"],
}.items():
    mean, low, high = paired_mean_interval(differences.to_numpy())
    bootstrap_rows.append({
        "group": label, "users": len(differences),
        "mean_gain_pp": 100 * mean, "ci_lower_pp": 100 * low, "ci_upper_pp": 100 * high,
        "split_id": SPLIT_ID, "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "test", "bootstrap_method": "paired user percentile", "seed": 42,
    })
gain_uncertainty = pd.DataFrame(bootstrap_rows)
gain_uncertainty.to_csv(processed_dir / "test_recall_gain_uncertainty.csv", index=False)
shown_uncertainty = gain_uncertainty.copy()
for column in ("mean_gain_pp", "ci_lower_pp", "ci_upper_pp"):
    shown_uncertainty[column] = shown_uncertainty[column].map("{:+.4f}".format)
display(shown_uncertainty)
